# UrduStack — Train Risk Scorer (LoRA XLM-RoBERTa on PURUTT)

Run this notebook in Google Colab free-tier GPU. It fine-tunes XLM-RoBERTa-base on a 10–15k sample of PURUTT using LoRA, then computes a temperature-scaling parameter on the validation set.

Before running: upload `PURUTT.csv` to Colab or mount Drive.

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q transformers datasets peft bitsandbytes accelerate evaluate scikit-learn scipy pandas

In [ ]:
# Optional: mount Google Drive so the trained adapter is saved permanently
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the UrduStack repo (or upload the training script manually)
!git clone https://github.com/munazat/UrduStack.git
%cd UrduStack

In [ ]:
# Upload PURUTT.csv if you have not placed it in Drive
from google.colab import files
import shutil, os
os.makedirs('data/raw', exist_ok=True)
if not os.path.exists('data/raw/PURUTT.csv'):
    print('Please upload PURUTT.csv')
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith('.csv'):
            shutil.move(name, 'data/raw/PURUTT.csv')
            break
else:
    print('PURUTT.csv already present')

In [ ]:
# Inspect the dataset
import pandas as pd
df = pd.read_csv('data/raw/PURUTT.csv')
print(df.head())
print(df.columns.tolist())
print(df['label'].value_counts())

In [ ]:
# Train
!python scripts/train_risk_model.py \
  --data_path data/raw/PURUTT.csv \
  --output_dir models/risk_lora \
  --max_samples 15000 \
  --val_samples 2000 \
  --test_samples 2000 \
  --num_epochs 3 \
  --batch_size 16

In [ ]:
# Check outputs
import os
print('Adapter files:', os.listdir('models/risk_lora'))
if os.path.exists('models/temperature.txt'):
    print('Temperature:', open('models/temperature.txt').read().strip())

In [ ]:
# Optional: push adapter to Hugging Face Hub
# !huggingface-cli login
# !python scripts/train_risk_model.py \
#   --data_path data/raw/PURUTT.csv \
#   --output_dir models/risk_lora \
#   --push_to_hub \
#   --hub_model_id your-username/urdustack-risk-lora

In [ ]:
# Optional: copy results to Drive
import shutil
shutil.copytree('models', '/content/drive/MyDrive/urdustack_models', dirs_exist_ok=True)
print('Copied models to Drive')